Split Dataset

In this notebook, I will split the raw Human Message Rewriter dataset into three files:

- train.jsonl
- validation.jsonl
- test.jsonl

The training set will be used for fine-tuning.
The validation set will be used during training/checking.
The test set will be kept separate for final evaluation.

# Importing Libraries

In [3]:

import json
from pathlib import Path
import random
from sklearn.model_selection import train_test_split

# Setup configurations

## Setup paths

In [4]:
raw_dataset_path= Path("../data/raw/human_rewriter_raw.jsonl")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"Raw dataset path: {raw_dataset_path} | Raw dataset exists: {raw_dataset_path.exists()}")
print(f"Processed directory: {processed_dir} | Exists: {processed_dir.exists()}")


Raw dataset path: ..\data\raw\human_rewriter_raw.jsonl | Raw dataset exists: True
Processed directory: ..\data\processed | Exists: True


In [5]:
train_path = processed_dir / "train.jsonl"
validation_path = processed_dir / "validation.jsonl"
test_path = processed_dir / "test.jsonl"

## Load raw dataset

In [6]:
rows = []

with raw_dataset_path.open("r",encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
len(rows)

300

# Random Split

## Shuffle the dataset

In [10]:
random.seed(42)
shuffled_rows = rows.copy()
random.shuffle(shuffled_rows)



## 80/10/10 split

In [12]:
total_rows = len(shuffled_rows)

train_size = int(total_rows * 0.8)
validation_size = int(total_rows * 0.1)



In [13]:
train_rows = shuffled_rows[:train_size]
validation_rows = shuffled_rows[train_size:train_size + validation_size]
test_rows = shuffled_rows[train_size + validation_size:]

print("Train rows:", len(train_rows))
print("Validation rows:", len(validation_rows))
print("Test rows:", len(test_rows))

Train rows: 240
Validation rows: 30
Test rows: 30


# Save JSONL

In [12]:
def save_jsonl(rows, file_path):
    with file_path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

In [13]:
save_jsonl(train_rows, train_path)
save_jsonl(validation_rows, validation_path)
save_jsonl(test_rows, test_path)

print("Saved:")
print(train_path)
print(validation_path)
print(test_path)

Saved:
..\data\processed\train.jsonl
..\data\processed\validation.jsonl
..\data\processed\test.jsonl


In [14]:
print("Train exists:", train_path.exists())
print("Validation exists:", validation_path.exists())
print("Test exists:", test_path.exists())

Train exists: True
Validation exists: True
Test exists: True


In [15]:
def count_jsonl_rows(file_path):
    with file_path.open("r", encoding="utf-8") as file:
        return sum(1 for line in file if line.strip())


print("Train rows:", count_jsonl_rows(train_path))
print("Validation rows:", count_jsonl_rows(validation_path))
print("Test rows:", count_jsonl_rows(test_path))

Train rows: 240
Validation rows: 30
Test rows: 30


## Count Category per split

In [16]:
def count_categories(rows):
    category_counts = {}

    for row in rows:
        category = row["category"]
        category_counts[category] = category_counts.get(category, 0) + 1

    return category_counts

In [17]:
train_category_counts = count_categories(train_rows)
validation_category_counts = count_categories(validation_rows)
test_category_counts = count_categories(test_rows)

print("Train categories:")
print(train_category_counts)

print("\nValidation categories:")
print(validation_category_counts)

print("\nTest categories:")
print(test_category_counts)

Train categories:
{'workplace': 23, 'reminder': 25, 'clearer_text': 29, 'awkward_casual': 29, 'formal_email': 25, 'polite_request': 26, 'wordy_to_concise': 32, 'student_message': 27, 'follow_up': 24}

Validation categories:
{'reminder': 3, 'student_message': 4, 'clearer_text': 3, 'formal_email': 2, 'workplace': 6, 'wordy_to_concise': 4, 'polite_request': 2, 'awkward_casual': 3, 'follow_up': 3}

Test categories:
{'workplace': 6, 'student_message': 4, 'awkward_casual': 3, 'clearer_text': 3, 'follow_up': 3, 'formal_email': 3, 'wordy_to_concise': 4, 'polite_request': 2, 'reminder': 2}


In [18]:
all_categories = sorted(count_categories(rows).keys())

print("Category distribution by split")
print("-" * 50)

for category in all_categories:
    train_count = train_category_counts.get(category, 0)
    validation_count = validation_category_counts.get(category, 0)
    test_count = test_category_counts.get(category, 0)

    print(
        f"{category:20} | "
        f"train: {train_count:3} | "
        f"validation: {validation_count:3} | "
        f"test: {test_count:3}"
    )

Category distribution by split
--------------------------------------------------
awkward_casual       | train:  29 | validation:   3 | test:   3
clearer_text         | train:  29 | validation:   3 | test:   3
follow_up            | train:  24 | validation:   3 | test:   3
formal_email         | train:  25 | validation:   2 | test:   3
polite_request       | train:  26 | validation:   2 | test:   2
reminder             | train:  25 | validation:   3 | test:   2
student_message      | train:  27 | validation:   4 | test:   4
wordy_to_concise     | train:  32 | validation:   4 | test:   4
workplace            | train:  23 | validation:   6 | test:   6


## Missing categories in each split

In [19]:
def find_missing_categories(split_counts, all_categories):
    missing = []

    for category in all_categories:
        if split_counts.get(category, 0) == 0:
            missing.append(category)

    return missing


missing_train_categories = find_missing_categories(train_category_counts, all_categories)
missing_validation_categories = find_missing_categories(validation_category_counts, all_categories)
missing_test_categories = find_missing_categories(test_category_counts, all_categories)

print("Missing categories in train:", missing_train_categories)
print("Missing categories in validation:", missing_validation_categories)
print("Missing categories in test:", missing_test_categories)

Missing categories in train: []
Missing categories in validation: []
Missing categories in test: []


In [20]:
print("Final Split Summary")
print("=" * 40)

print(f"Raw dataset rows: {len(rows)}")
print(f"Train rows: {len(train_rows)}")
print(f"Validation rows: {len(validation_rows)}")
print(f"Test rows: {len(test_rows)}")

print("\nMissing category check:")
print(f"Train missing categories: {len(missing_train_categories)}")
print(f"Validation missing categories: {len(missing_validation_categories)}")
print(f"Test missing categories: {len(missing_test_categories)}")

if (
    len(missing_train_categories) == 0
    and len(missing_validation_categories) == 0
    and len(missing_test_categories) == 0
):
    print("\nStatus: PASSED")
    print("All splits contain examples from every category.")
else:
    print("\nStatus: REVIEW NEEDED")
    print("At least one split is missing a category.")

Final Split Summary
Raw dataset rows: 300
Train rows: 240
Validation rows: 30
Test rows: 30

Missing category check:
Train missing categories: 0
Validation missing categories: 0
Test missing categories: 0

Status: PASSED
All splits contain examples from every category.


This split tells us that the random split is not very useful. We might need to replace the simple random split with a stratified split. 
Stratified split means keeping roughly the same category distribution in train, validation, and test.

# Stratified sampling

In [7]:
random.seed(42)
shuffled_rows = rows.copy()
random.shuffle(shuffled_rows)

In [8]:
shuffled_rows[:5]

[{'id': 'ex_0107',
  'category': 'follow_up',
  'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.',
  'input': 'I wanted to follow up because I have not received a response yet.',
  'output': 'I wanted to follow up since I haven’t received a response yet.',
  'source': 'synthetic_curated'},
 {'id': 'ex_0240',
  'category': 'awkward_casual',
  'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.',
  'input': 'I was wondering that maybe we should divide the work again.',
  'output': 'Maybe we should divide the work again.',
  'source': 'synthetic_curated'},
 {'id': 'ex_0038',
  'category': 'student_message',
  'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.',
  'input': 'I apologize for the inconvenience, but I will not be able to submit the assignment by the original deadline.',
  'output': 'I’m sorry, but I won’t be

In [9]:
categories = [row["category"] for row in shuffled_rows]

train_rows, temp_rows = train_test_split(
    rows,
    test_size = 0.2,
    random_state = 42,
    stratify = categories
)

len(train_rows), len(temp_rows)

(240, 60)

In [10]:
temp_categories = [row["category"] for row in temp_rows]

validation_rows, test_rows = train_test_split(
    temp_rows,
    test_size = 0.5,
    random_state = 42,
    stratify = temp_categories
)

len(validation_rows), len(test_rows)

(30, 30)

In [11]:


print("Train rows:", len(train_rows))
print("Validation rows:", len(validation_rows))
print("Test rows:", len(test_rows))

Train rows: 240
Validation rows: 30
Test rows: 30


Stratified Split
- The first random split caused the validation set to miss one category.
- To fix this, we used a stratified split based on the `category` column. This keeps the category distribution more balanced across train, validation, and test sets.
- This matters because our evaluation should include different types of rewriting examples, not just a subset of styles.